In [1]:
import pandas as pd
import string
import re
import unicodedata
import demoji

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN
from imblearn.under_sampling import RandomUnderSampler


In [2]:
#this will read the csv file and seperate it into two columns
# Text and Class (being Non_Hope Speach or Hope Speech)
#we then print the dataframe to see the contents.
df = pd.read_csv("english_hope_train _comma.csv",sep=',', names=['Text', 'Class'])
print(df)


                                                    Text            Class
0      these tiktoks radiate gay chaotic energy and i...  Non_hope_speech
1      @Champions Again He got killed for using false...  Non_hope_speech
2                   It's not that all lives don't matter  Non_hope_speech
3      Is it really that difficult to understand? Bla...  Non_hope_speech
4      Whenever we say black isn't that racists?  Why...  Non_hope_speech
...                                                  ...              ...
22757  It's a load of bollocks every life matters sim...  Non_hope_speech
22758  no say it because all lives matter! deku would...  Non_hope_speech
22759                          God says her life matters  Non_hope_speech
22760  This video is just shit. A bunch of whiny ass ...  Non_hope_speech
22761  Mc Fortnut2821 she did 4 months ago in west ch...  Non_hope_speech

[22762 rows x 2 columns]


In [3]:

#this code will remove usernames, using the built in (re) library.
df['Text'] = df['Text'].str.replace(r'@\w+', '',regex=True)

#This next line removes the Puntuation from the data
#we create a variable that will specfiy what we want to remove in the text. 
#we then use puncuation_Removal variable to remove the punctuation from the text.
puncuation_Removal = r"[{}“”‘’\"\'—–]".format(re.escape(string.punctuation))
df['Text'] = df['Text'].str.replace(puncuation_Removal, '', regex=True)


#This code replaces emojis in the 'Text' column with their descriptions utilizing the demoji library.
df['Text'] = df['Text'].apply(demoji.replace_with_desc)


#Normalize the Unicode to strip stylized/custom fonts
df['Text'] = df['Text'].apply(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode())


#Remove single character words 
df['Text'] = df['Text'].str.replace(r'\b\w\b', '', regex=True)


#Removes the whitespaces that are left after our data processing steps.
df['Text'] = df['Text'].str.strip().str.replace(r'\s+', ' ', regex=True)


#prints parts of our data set to test out if our code succefully cleaned the data. )
print(df.head(71))

print(df.head(87))

                                                 Text            Class
0   these tiktoks radiate gay chaotic energy and l...  Non_hope_speech
1           Again He got killed for using false money  Non_hope_speech
2                  Its not that all lives dont matter  Non_hope_speech
3   Is it really that difficult to understand Blac...  Non_hope_speech
4   Whenever we say black isnt that racists Why do...  Non_hope_speech
..                                                ...              ...
66  would have asked him if hes watched but Im che...  Non_hope_speech
67  feel so base for that guy They treated him as ...      Hope_speech
68                              American Lives Matter  Non_hope_speech
69  :thinking face:staged set up for the coming ra...  Non_hope_speech
70                             The interviewer is HOT  Non_hope_speech

[71 rows x 2 columns]
                                                 Text            Class
0   these tiktoks radiate gay chaotic energy and l... 

In [4]:
#now we apply linear Regression to the data set. (we have seen online that logistic regression is better for binary classification)
#Our Class column has values that make it a multiclass data set. to apply linear regression we need to convert it to a binary data set.
#we will show the unique values with the following code.
print(df["Class"].value_counts()) #counts the number of unique values in the Class column
print(df["Class"].isnull().sum()) #Checks for any null values in the Class column

#now we will remove any rows that are empty or have the unwanted values
df_clean = df.dropna()
#only keeps rows that have the values "Non_hope_speech" or "Hope_speech" in the Class column
df_clean = df_clean[df_clean["Class"].isin(["Non_hope_speech", "Hope_speech"])]


#we store the "Text" column into the dataframe X 
X = df_clean.drop("Text", axis=1)
#We store the "Class" column into the dataframe y and convert the values to binary
y = df_clean["Class"].map({"Non_hope_speech": 0, "Hope_speech": 1,})  

#this will convert the text data into a matrix of token counts. 
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df_clean["Text"])

#this will show us our cleaned data set ready for linear regression.
print("\n-----------After Cleaning--------- \n")
print(y.unique())
print(df_clean["Class"].value_counts())

#saving the cleaned dataframe to a new csv file
df.to_csv("cleaned_english_hope_train_comma.csv", index=False)

Class
Non_hope_speech                                                                     20700
Hope_speech                                                                          1945
not-English                                                                            22
)                                                                                       7
-)                                                                                      5
                                                                                    ...  
p                                                                                       1
s my opinion.                                                                           1
one that has killed 428                                                                 1
the same horrible generalization that befell the entirety of the Germanic nation        1
they always come home to roost                                                          1
Name

In [5]:
#----------------------------------------------------------------------------Splitting Data & Kfold----------------------------------------------------------
#Now we will split the data into training and testing sets. 
#test_size=0.2 means 20% of the data will be used for testing and 80% for training and eval.
#random state=50 is used to ensure reproducibility of the results.
X_train_Eval, X_test, y_train_Eval, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

#to add an evauation split we will take the 80% of the train data and split 12.5 (10% of totatl data)
X_train, X_eval, y_train, y_eval = train_test_split(X_train_Eval, y_train_Eval, test_size=0.125, random_state=50)

#K-Fold cross-validation
#n_splits=10 means the data will be split into 10 parts/folds.
#shuffle=True means the data will be shuffled before splitting.
#cross_val is primarily used to train and evaluate the model DO NOT USE ON TEST
k_folds = KFold(n_splits=10, shuffle= True, random_state=50) 

In [ ]:
#----------------------------------------------------------------------------SMOTE BALANCING----------------------------------------------------------
#Generating synthetic samples for the minority class to balance the dataset
#Offers more diverse minority data, reduces overfitting risk
#improves generalization of the model

# Apply SMOTE to the training data
smote = SMOTE(random_state=50)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

#F1 score With LR = .88

In [ ]:
#----------------------------------------------------------------------------ADASYN BALANCING----------------------------------------------------------
#Generates synthetic samples for the minority class based on data distribution
#Focuses on harder-to-learn examples, improving model robustness
#improves decision boundary for complex datasets and reduces bias

#ADASYN is an extension of SMOTE that adaptively generates synthetic samples for the minority class based on the density of the data points.
#sampling_strategy='auto' means it will balance the classes automatically.
#random_state=50 is used to ensure reproducibility of the results.
#n_neighbors=5 means it will consider 5 nearest neighbors to generate synthetic samples. higher values can lead to more generalized samples.

adasyn = ADASYN(sampling_strategy='auto', random_state=50, n_neighbors=5)
X_train_bal, y_train_bal = adasyn.fit_resample(X_train, y_train)

#F1 score with LR = .79

In [ ]:
#----------------------------------------------------------------------------Random Over Sampling BALANCING----------------------------------------------------------
#Prevent bias toward majority class, ensure classifier learns from both classes
#fast and easy: duplicates real samples doesn't generate new samples

# Apply Random Over Sampling to the training data
r_over_sampler = RandomOverSampler(random_state=50)
X_train_bal, y_train_bal = r_over_sampler.fit_resample(X_train, y_train)

#F1 score With LR = .94

In [ ]:
#----------------------------------------------------------------------------Random Under Sampling BALANCING----------------------------------------------------------
#Prevent bias toward majority class: reduces size of majority class to match minority class
#Fast and memoru efficient: deletes samples from majority class using less RAM
#doesn't touch minority class.
#ISSUES: loss of data, potential underfitting (could weaken model)

# Apply Random under Sampling to the training data
r_under_sampler = RandomUnderSampler(random_state=50)
X_train_bal, y_train_bal = r_under_sampler.fit_resample(X_train, y_train)

#F1 score With LR = .89

In [7]:
#----------------------------------------------------------------------------logistic regression----------------------------------------------------------
#max_iter is the maximum number of iterations the solver will use to find the best weights for the model.
#random_state is used to ensure reproducibility of the results.
logR = LogisticRegression(max_iter=1000, random_state=50)


#cross_val_score will evaluate the logistic regression model using K-Fold cross-validation.
#cross_val is primarily used to train and evaluate the model DO NOT USE ON TEST
scores = cross_val_score(logR, X_train_bal, y_train_bal, cv=k_folds, scoring= 'accuracy')

#Output results
#prints the accuracy for each fold, the average accuracy across all folds, and the number of folds used.
print("Cross Validation Scores: ", scores) 
print("Average CV Score: ", scores.mean()) 
print("Number of CV Scores used in Average: ", len(scores)) 


#We use the code from before to train the logistic regression model.
#trains the model on our data
logR.fit(X_train_bal, y_train_bal)

#Will make predictions on the Evaluation data. #Convert the predictions to binary values by using a threshold of 0.5
y_eval_pred = logR.predict(X_eval)
y_eval_pred_binary = (y_eval_pred >= 0.5).astype(int)

#Now we print the accuracy. 
print("\n--------LR Evaluation Results--------\n")
print("Accuracy:", accuracy_score(y_eval_pred, y_eval_pred_binary))
print("Confusion Matrix:\n", confusion_matrix(y_eval_pred, y_eval_pred_binary))
print("Classification Report:\n", classification_report(y_eval_pred, y_eval_pred_binary))

#Will make predictions on the test data. #Convert the predictions to binary values by using a threshold of 0.5
y_test_pred = logR.predict(X_test)
y_test_pred_binary = (y_test_pred >= 0.5).astype(int)

#Now we print the accuracy. 
print("\n--------LR Final Test Results--------\n")
print("Accuracy:", accuracy_score(y_test, y_test_pred_binary))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred_binary))
print("Classification Report:\n", classification_report(y_test, y_test_pred_binary))

Cross Validation Scores:  [0.87945493 0.88120196 0.87945493 0.87281621 0.89133473 0.86408106
 0.87351502 0.87002096 0.87836421 0.87102412]
Average CV Score:  0.8761268130092859
Number of CV Scores used in Average:  10

--------LR Evaluation Results--------

Accuracy: 1.0
Confusion Matrix:
 [[1792    0]
 [   0  473]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1792
           1       1.00      1.00      1.00       473

    accuracy                           1.00      2265
   macro avg       1.00      1.00      1.00      2265
weighted avg       1.00      1.00      1.00      2265


--------LR Final Test Results--------

Accuracy: 0.7873702804151027
Confusion Matrix:
 [[3365  760]
 [ 203  201]]
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.82      0.87      4125
           1       0.21      0.50      0.29       404

    accuracy                

In [ ]:
#----------------------------------------------------------------------------logistic regression----------------------------------------------------------
#max_iter is the maximum number of iterations the solver will use to find the best weights for the model.
#random_state is used to ensure reproducibility of the results.
Rforest = RandomForestClassifier(max_iter=1000, random_state=50)


#cross_val_score will evaluate the logistic regression model using K-Fold cross-validation.
#cross_val is primarily used to train and evaluate the model DO NOT USE ON TEST
scores = cross_val_score(logR, X_train_bal, y_train_bal, cv=k_folds, scoring= 'accuracy')

#Output results
#prints the accuracy for each fold, the average accuracy across all folds, and the number of folds used.
print("Cross Validation Scores: ", scores) 
print("Average CV Score: ", scores.mean()) 
print("Number of CV Scores used in Average: ", len(scores)) 


#We use the code from before to train the logistic regression model.
#trains the model on our data
Rforest.fit(X_train_bal, y_train_bal)

#Will make predictions on the Evaluation data. #Convert the predictions to binary values by using a threshold of 0.5
y_eval_pred = Rforest.predict(X_eval)
y_eval_pred_binary = (y_eval_pred >= 0.5).astype(int)

#Now we print the accuracy. 
print("\n--------LR Evaluation Results--------\n")
print("Accuracy:", accuracy_score(y_eval_pred, y_eval_pred_binary))
print("Confusion Matrix:\n", confusion_matrix(y_eval_pred, y_eval_pred_binary))
print("Classification Report:\n", classification_report(y_eval_pred, y_eval_pred_binary))

#Will make predictions on the test data. #Convert the predictions to binary values by using a threshold of 0.5
y_test_pred = Rforest.predict(X_test)
y_test_pred_binary = (y_test_pred >= 0.5).astype(int)

#Now we print the accuracy. 
print("\n--------LR Final Test Results--------\n")
print("Accuracy:", accuracy_score(y_test, y_test_pred_binary))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred_binary))
print("Classification Report:\n", classification_report(y_test, y_test_pred_binary))